---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 3
python-dotenv could not parse statement starting at line 6
python-dotenv could not parse statement starting at line 9


Root project: d:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [3]:
import pandas as pd
import random

corpus = pd.read_json("D:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2\data\cleaned\corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[NicusorDanRO] Foarte bine 👍, va rog sa trimiteti un camion de xanax in rusia, sa nu se apuce d
[@CălinGeorgescu-CanalulOficial] Si mata esti omul sistemului pt ca toata viata ai lucrat cu sistemul.Noi cei car
[turcescu111] Nu tin apararea nimanui, dar atata timp cat avem parteneriat cu SUA, ei ne-au ce


<>:4: SyntaxWarning: invalid escape sequence '\A'
<>:4: SyntaxWarning: invalid escape sequence '\A'
C:\Users\crist\AppData\Local\Temp\ipykernel_32592\2965279881.py:4: SyntaxWarning: invalid escape sequence '\A'
  corpus = pd.read_json("D:\ADC_2024\An 2\Sem II\Ingineria AI\echochamber-project-team-2\data\cleaned\corpus_youtube_sample.jsonl", lines=True)


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte

AXA_1 = "nationalism"
AXA_2 = "anti_elite"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
nationalism măsoară gradul în care comentariul exprimă idei naționaliste,
patriotice sau pune accent pe identitatea națională românească.

0 = absent
Comentariul nu conține referințe naționaliste sau patriotice.

1 = prezent
Comentariul conține referințe moderate la patriotism, țară,
națiune sau valori naționale.

2 = dominant
Mesajul este puternic centrat pe identitate națională,
suveranitate, patriotism sau opoziție față de influențe externe.
"""

AXA_2_DEFINITION = """
anti_elite măsoară gradul în care comentariul exprimă ostilitate,
neîncredere sau opoziție față de elite politice, media,
instituții sau grupuri percepute ca elite.

0 = absent
Comentariul nu critică elitele sau instituțiile.

1 = prezent
Comentariul exprimă critică moderată sau neîncredere
față de elite sau instituții.

2 = dominant
Comentariul este puternic anti-elitist și prezintă elitele
ca responsabile pentru probleme sociale sau politice.
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [6]:
MINI_PROMPT = f"""
Ești un analist de discurs politic online.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un analist de discurs politic online.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. nationalism
2. anti_elite
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
nationalism = 0 / 1 / 2
anti_elite = 0 / 1 / 2
DEFINIȚII:

nationalism măsoară gradul în care comentariul exprimă idei naționaliste,
patriotice sau pune accent pe identitatea națională românească.

0 = absent
Comentariul nu conține referințe naționaliste sau patriotice.

1 = prezent
Comentariul conține referințe moderate la patriotism, țară,
națiune sau valori naționale.

2 = dominant
Mesajul este puternic centrat pe identitate națională,
suveranitate, patriotism sau opoziție față de influențe externe.


anti_elite măsoară gradul în care comentariul exprimă ostilitate,
neîncredere sau opoziție față de elite politice, media,
instituții sau gru

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [8]:
TESTS = corpus.sample(5, random_state=33)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
79,yt_8tryQi2tezs_UgzjPmN9BMHOdrUDqFB4AaABAg,georgesimionoficial,Respect pentru muncă! În dialog cu poporul rom...,Hrană. -Apa- energie Sunt lucruri de bază de l...
98,yt_HDsCdSOtxO0_Ugy7qtu7Bhf22Mk-tFx4AaABAg,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,"Mereu la înălțimea așteptărilor noastre,sunteț..."
34,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,europafmromania,Judecata de Joi - 02.04.2026,"Catu sa plateasca, daca nu se pruc probe ale u..."
241,yt_jvV4hfEOyQM_UgzCU2NKMQhbkm-0P914AaABAg,georgesimionoficial,Am filmat acest material acum o săptămână la S...,Păi ați refuzat dialogul la televiziune. Cum a...
0,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,NicusorDanRO,🟢 LIVE Declarații de presă susținute la Palatu...,jigodia aia de Georgescu nu lua intrebari inca...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [9]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [10]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [ ]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Hrană. -Apa- energie Sunt lucruri de bază de la care pleacă toată puterea economică a unei țări

OUTPUT MODEL:
```json
{
  "target": "none",
  "stance": "none",
  "tone": "neutru",
  "nationalism": 1,
  "anti_elite": 0
}
```
COMENTARIU:
Mereu la înălțimea așteptărilor noastre,sunteți voi Recorder!Felicitări vouă!Atâta timp cât mai există o rază de speranță în țărișoara asta lupta nu trebuie încheiată.Curaj celor drepți!

OUTPUT MODEL:
```json
{
  "target": "Recorder Romania",
  "stance": "pro",
  "tone": "mobilizator",
  "nationalism": 1,
  "anti_elite": 0
}
```
COMENTARIU:
Catu sa plateasca, daca nu se pruc probe ale unei decizii la nivel UE. Si nici decixcizia apriorica de reducere a comenzii bazate pe estimarea corecta a numarului imbecililor antivaccinisti romani nu era posibila. Voiculescu nu, ca s-a opus faptic si oficial deciziei de procurare.

OUTPUT MODEL:
```json
{
  "target": "Catu",
  "stance": "anti",
  "tone": "acuzator",
  "nationalism": 0,
  "anti_elite": 1


In [13]:
import re

cleaned_results = []

for item in results:

    # elimină ```json și ```
    cleaned = re.sub(
        r"```json|```",
        "",
        item["model_output"]
    ).strip()

    try:
        parsed = json.loads(cleaned)

        cleaned_results.append({
            "id": item["id"],
            "text": item["text"],
            "target": parsed.get("target"),
            "stance": parsed.get("stance"),
            "tone": parsed.get("tone"),
            "nationalism": parsed.get("nationalism"),
            "anti_elite": parsed.get("anti_elite")
        })

        print(f"JSON VALID pentru {item['id']}")

    except json.JSONDecodeError as e:

        print(f"JSON INVALID pentru {item['id']}")
        print(e)

results_df = pd.DataFrame(cleaned_results)

results_df

JSON VALID pentru yt_8tryQi2tezs_UgzjPmN9BMHOdrUDqFB4AaABAg
JSON VALID pentru yt_HDsCdSOtxO0_Ugy7qtu7Bhf22Mk-tFx4AaABAg
JSON VALID pentru yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg
JSON VALID pentru yt_jvV4hfEOyQM_UgzCU2NKMQhbkm-0P914AaABAg
JSON VALID pentru yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg


,id,text,target,stance,tone,nationalism,anti_elite
0,yt_8tryQi2tezs_UgzjPmN9BMHOdrUDqFB4AaABAg,Hrană. -Apa- energie Sunt lucruri de bază de l...,none,none,neutru,1,0
1,yt_HDsCdSOtxO0_Ugy7qtu7Bhf22Mk-tFx4AaABAg,"Mereu la înălțimea așteptărilor noastre,sunteț...",Recorder Romania,pro,mobilizator,1,0
2,yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg,"Catu sa plateasca, daca nu se pruc probe ale u...",Catu,anti,acuzator,0,1
3,yt_jvV4hfEOyQM_UgzCU2NKMQhbkm-0P914AaABAg,Păi ați refuzat dialogul la televiziune. Cum a...,diaspora,anti,acuzator,1,0
4,yt_Vekhmz5OPCc_UgzpXOYhxlT3Nqo6h7J4AaABAg,jigodia aia de Georgescu nu lua intrebari inca...,Georgescu,anti,acuzator,0,1


In [14]:
cleaned_results

[{'id': 'yt_8tryQi2tezs_UgzjPmN9BMHOdrUDqFB4AaABAg',
  'text': 'Hrană. -Apa- energie Sunt lucruri de bază de la care pleacă toată puterea economică a unei țări',
  'target': 'none',
  'stance': 'none',
  'tone': 'neutru',
  'nationalism': 1,
  'anti_elite': 0},
 {'id': 'yt_HDsCdSOtxO0_Ugy7qtu7Bhf22Mk-tFx4AaABAg',
  'text': 'Mereu la înălțimea așteptărilor noastre,sunteți voi Recorder!Felicitări vouă!Atâta timp cât mai există o rază de speranță în țărișoara asta lupta nu trebuie încheiată.Curaj celor drepți!',
  'target': 'Recorder Romania',
  'stance': 'pro',
  'tone': 'mobilizator',
  'nationalism': 1,
  'anti_elite': 0},
 {'id': 'yt_cok3YTTn8sg_Ugw9joa6s_uddLOSdCZ4AaABAg',
  'text': 'Catu sa plateasca, daca nu se pruc probe ale unei decizii la nivel UE. Si nici decixcizia apriorica de reducere a comenzii bazate pe estimarea corecta a numarului imbecililor antivaccinisti romani nu era posibila. Voiculescu nu, ca s-a opus faptic si oficial deciziei de procurare.',
  'target': 'Catu',
 

## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
- De ce le-ai ales?
- Modelul a returnat JSON corect?
- Care a fost cea mai mare problemă?
- Ce ai schimba în prompt?

## Interpretare și mini-tipologie

Am ales două axe de analiză:
- nationalism
- anti_elite

Aceste axe permit observarea modului în care comentariile politice combină discursul naționalist cu atitudinile anti-establishment.

Modelul a returnat JSON valid pentru exemplele analizate.

Pe baza celor două axe, poate fi construită o mini-tipologie:

- nationalism scăzut + anti_elite scăzut:
  comentarii neutre sau informative

- nationalism ridicat + anti_elite scăzut:
  comentarii predominant patriotice sau suveraniste

- nationalism scăzut + anti_elite ridicat:
  comentarii anti-sistem sau anti-politicieni

- nationalism ridicat + anti_elite ridicat:
  comentarii populist-naționaliste

Rezultatele arată că modelul poate identifica teme ideologice generale din comentarii politice scurte, însă comentariile ironice sau ambigue rămân mai dificil de clasificat.